<a href="https://colab.research.google.com/github/Hanzet22/Ipeye-enbe/blob/main/fetch_dns_pool_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DNS IP Pool Fetcher v4 (By Claude)
Scrape publicdns.xyz max 20 halaman per negara → export JSON

| Negara | Estimasi IP |
|--------|-------------|
| JP, TW, SG, ID, US, CA, FR, TR, PL, KR, DE, BR, TH, GB | max 20 pages each |

**Estimasi waktu: ~7 menit**

In [ ]:
# Cell 1 — Install
!pip install requests beautifulsoup4 -q
print('Done')

In [ ]:
# Cell 2 — Imports & Config
import requests
from bs4 import BeautifulSoup
import re
import time
import json

BASE    = 'https://www.publicdns.xyz/country'
HEADERS = {'User-Agent': 'Mozilla/5.0 Chrome/124.0'}
DELAY   = 1.0
MAX_PAGE = 20

COUNTRIES = [
    'jp', 'tw', 'sg', 'id',
    'us', 'ca', 'fr', 'tr',
    'pl', 'kr', 'de', 'br',
    'th', 'gb'
]

IPV4 = re.compile(r'^(\d{1,3}\.){3}\d{1,3}$')

print('Config ready')
print(f'Countries : {COUNTRIES}')
print(f'Max pages : {MAX_PAGE} per country')
print(f'Delay     : {DELAY}s')
print(f'Est. time : ~{int(MAX_PAGE * len(COUNTRIES) * DELAY / 60)} min')

In [ ]:
# Cell 3 — Scraper Functions

def fetch_page(cc, page):
    """Fetch satu halaman, return list IP atau [] kalau gagal/kosong"""
    url = f'{BASE}/{cc}.html' if page == 1 else f'{BASE}/{cc}/{page}.html'
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            return []
        soup = BeautifulSoup(r.text, 'html.parser')
        table = soup.find('table')
        if not table:
            return []
        ips = []
        for row in table.find_all('tr')[1:]:
            cells = row.find_all('td')
            if not cells:
                continue
            # Ambil token pertama — bisa 'IP  hostname', kita ambil IP-nya aja
            raw = cells[0].get_text(' ', strip=True).split()[0]
            if IPV4.match(raw):
                ips.append(raw)
        return ips
    except Exception as e:
        print(f'  ERR [{cc}] p{page}: {e}')
        return []


def scrape_country(cc, max_page=MAX_PAGE):
    """Scrape cc sampai max_page atau sampai halaman kosong"""
    pool, seen = [], set()
    print(f'\n[{cc.upper()}]')
    for page in range(1, max_page + 1):
        ips = fetch_page(cc, page)
        if not ips:
            print(f'  p{page} empty — stop')
            break
        added = 0
        for ip in ips:
            if ip not in seen:
                seen.add(ip)
                pool.append(ip)
                added += 1
        print(f'  p{page:>3} +{added:<3} -> {len(pool)}')
        if page < max_page:
            time.sleep(DELAY)
    print(f'  DONE: {len(pool)} IPs')
    return pool


print('Scraper ready')

In [ ]:
# Cell 4 — RUN (~7 menit)
POOL = {}
for cc in COUNTRIES:
    POOL[cc.upper()] = scrape_country(cc)
    time.sleep(DELAY * 2)

print('\n=== DONE ===')
for cc, ips in POOL.items():
    print(f'  {cc}: {len(ips)} IPs')
print(f'  TOTAL: {sum(len(v) for v in POOL.values())} IPs')

In [ ]:
# Cell 5 — Export JSON
import datetime

OUT = 'dns_pool.json'
output = {
    'meta': {
        'source'    : 'publicdns.xyz',
        'generated' : datetime.datetime.utcnow().isoformat() + 'Z',
        'max_pages' : MAX_PAGE,
        'total'     : sum(len(v) for v in POOL.values()),
        'countries' : {cc: len(ips) for cc, ips in POOL.items()}
    },
    'pool': POOL
}

with open(OUT, 'w') as f:
    json.dump(output, f, indent=2)

print(f'Saved: {OUT}')
print(f'Size : {len(json.dumps(output))/1024:.1f} KB')

from google.colab import files
files.download(OUT)
print('Download triggered!')